# Evaluating the LLM judge

Run the LLM judge (`llm_judge.py`) on every labeled run and compare its labels with
the human labels from `label_evals.py`. If the judge agrees with us, we can use it to
label new eval runs automatically.

Run this notebook from the `evals/` folder. Requires `OPENAI_API_KEY` in the project's `.env`.

In [19]:
from dotenv import load_dotenv

load_dotenv()

True

In [20]:
from llm_judge import create_log_judge_agent, format_judge_prompt

In [22]:
import json

with open("evals_run_2026_11_09_results.json", "r") as f:
    results = json.load(f)

print(f"Loaded {len(results)} results.")

Loaded 20 results.


In [23]:
judge = create_log_judge_agent()

## One run first

In [24]:
row = results[0]

In [25]:
prompt = format_judge_prompt(row)
print(prompt)

Evaluate the agent's performance for the following interaction.

User Question:
What processor does the Alexy G26 Pro use?

Tools Used:
search_docs({"query": "processor", "model": "Alexy G26 Pro"})

Agent Response:
The Alexy G26 Pro is equipped with the **Helios G1 Neural Chip**, which features a 4nm architecture.

Sources cited by the agent: Alexy_G26_Pro_GrigoTel_Product_Specifications.pdf
Answer type: product_specs


In [26]:
eval_result = await judge.run(prompt)
eval_output = eval_result.output

In [27]:
print(eval_output.label)
print(eval_output.reasoning)

good
1. The claim states that the Alexy G26 Pro is equipped with the Helios G1 Neural Chip, which is correct according to the reference material. 
2. The agent also specifies that the chip features a 4nm architecture, which is supported by the specifications document for the Alexy G26 Pro. 
3. The agent used the appropriate tool (search_docs) to retrieve this information, retrieving the correct source document. 
4. All information provided in the agent's response is accurate, directly sourced, and aligns with the details in the product specifications. 
5. The response does not contain any made-up content and answers the user’s question appropriately.

Overall, the agent's response meets all criteria for a good response. 

Final Evaluation: good


## All runs

In [28]:
async def run_eval(row):
    prompt = format_judge_prompt(row)
    eval_result = await judge.run(prompt)
    eval_output = eval_result.output
    return eval_output

In [29]:
from tqdm.auto import tqdm

In [30]:
eval_results = []

for row in tqdm(results):
    eval_output = await run_eval(row)
    eval_results.append({
        'row': row,
        'label': eval_output.label,
        'reasoning': eval_output.reasoning
    })

  0%|          | 0/20 [00:00<?, ?it/s]

## Judge vs. human labels

In [31]:
rows = []

for result in eval_results:
    row = {
        'question': result['row']['input']['question'],
        'type': result['row']['input']['type'],
        'llm_answer': result['row']['response']['answer'],
        'human_label': result['row']['label'],
        'human_comment': result['row']['comments'],
        'llm_label': result['label'],
        'llm_reasoning': result['reasoning']
    }
    rows.append(row)

In [32]:
import pandas as pd
pd.set_option('display.max_colwidth', 120)

df_rows = pd.DataFrame(rows)

In [33]:
df_rows.llm_label.value_counts()

llm_label
good    10
bad     10
Name: count, dtype: int64

In [36]:
df_rows.human_label.value_counts()

human_label
good    15
bad      5
Name: count, dtype: int64

Agreement: how often the judge gives the same label as the human:

In [35]:
(df_rows.llm_label == df_rows.human_label).mean()

np.float64(0.65)

Where they agree and disagree. *bad → good* means a human said "bad" but the judge said
"good": the judge missed a problem. That's the more dangerous mistake for a judge.

In [17]:
pd.crosstab(df_rows.human_label, df_rows.llm_label, rownames=['human'], colnames=['judge'])

judge,bad,good
human,,
bad,4,1
good,5,10


In [18]:
df_disagreement = df_rows[df_rows.llm_label != df_rows.human_label]

In [19]:
df_disagreement[['question', 'human_label', 'llm_label', 'human_comment']]

,question,human_label,llm_label,human_comment
1,Does the G26 Lite have a fingerprint reader?,good,bad,
4,Does the G26 support wireless charging?,good,bad,
6,And how much does it cost?,good,bad,
9,I want to buy 2 Alexy G26 Fold phones,good,bad,exactly as per instruction in prompt.
10,I'd like to order 5 green G26 Fold,good,bad,very accurate as per instruction the AI suggested other options to the client.
16,Are you planning to release a tablet?,good,bad,
17,Does my fingerprint data get uploaded to the cloud?,bad,good,privacy file didn't talk about this fingerprint data.


Read the judge's reasoning for each disagreement:

In [20]:
for _, r in df_disagreement.iterrows():
    print(f"QUESTION: {r.question}")
    print(f"HUMAN: {r.human_label} ({r.human_comment or 'no comment'})")
    print(f"JUDGE: {r.llm_label}")
    print(r.llm_reasoning)
    print('-' * 80)

QUESTION: Does the G26 Lite have a fingerprint reader?
HUMAN: good (no comment)
JUDGE: bad
1. The user asked whether the G26 Lite has a fingerprint reader.  
2. The agent utilized the appropriate tool to search the documents for information related to the G26 Lite and the fingerprint reader.  
3. According to the specification sheet for the G26 Lite, the device features a biometric 'Vein-Scan' under-display sensor, not a traditional fingerprint reader.  
4. The agent incorrectly stated that they could not find specific information regarding a fingerprint reader, thus not providing accurate information based on the reference material.  
5. The agent failed to cite the correct terminology (Vein-Scan) instead of the user's term (fingerprint) and did not clarify that the information about the biometric sensor was present in the documents.
6. By not providing the correct information or its source, the agent's response is incomplete and misleading.  
Overall, while the reference material was

Save the judge's labels next to the human ones, for later comparison:

In [21]:
with open("evals_run_2026_11_09_judge.json", "w") as f_out:
    json.dump(rows, f_out, indent=2, ensure_ascii=False)